In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import re
import sqlite3
import random
import datetime
import logging
from crewai import Agent, Task, Crew, LLM


In [2]:

# ============================================================
# DATABASE SETUP
# ============================================================

conn = sqlite3.connect("support.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS support_tickets (
    ticket_id TEXT PRIMARY KEY,
    customer_name TEXT,
    message TEXT,
    status TEXT,
    created_at TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS interaction_logs (
    timestamp TEXT,
    customer_name TEXT,
    message TEXT,
    classification TEXT,
    agent_routed TEXT,
    success INTEGER
)
""")

conn.commit()

print("Database initialized successfully.")


Database initialized successfully.


In [3]:

# ============================================================
# LLM CONFIGURATION
# ============================================================

groq_key = os.getenv("GROQ_API_KEY")

if not groq_key:
    raise ValueError("GROQ_API_KEY not set in environment variables")

llm = LLM(
    model="meta-llama/llama-4-maverick-17b-128e-instruct",
    api_key=groq_key,
    base_url="https://api.groq.com/openai/v1",
    temperature=0.2
)

print("LLM configured successfully.")


LLM configured successfully.


In [4]:

# ============================================================
# AGENT DEFINITIONS
# ============================================================

classifier_agent = Agent(
    role="Classifier Agent",
    goal="Classify user input into Positive Feedback, Complaint, Ticket Status Query, or General Banking Query.",
    backstory="Expert in intent detection and sentiment analysis.",
    llm=llm
)

feedback_agent = Agent(
    role="Feedback Handler Agent",
    goal="Generate empathetic and professional responses for feedback.",
    backstory="Expert in customer relationship communication.",
    llm=llm
)

query_agent = Agent(
    role="Query Handler Agent",
    goal="Provide ticket status updates clearly and accurately.",
    backstory="Expert in ticket management systems.",
    llm=llm
)

print("Agents initialized.")


Agents initialized.


In [5]:

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def generate_ticket():
    while True:
        ticket = str(random.randint(100000, 999999))
        cursor.execute("SELECT * FROM support_tickets WHERE ticket_id=?", (ticket,))
        if not cursor.fetchone():
            return ticket

def classify_message(message):
    task = Task(
        description=f"""
Classify the message into one of:
1. Positive Feedback
2. Complaint
3. Ticket Status Query
4. General Banking Query

Message:
{message}

Return ONLY the category name.
""",
        expected_output="One category name",
        agent=classifier_agent
    )
    crew = Crew(agents=[classifier_agent], tasks=[task])
    return crew.kickoff().raw.strip()

def handle_positive_feedback(name):
    task = Task(
        description=f"Generate a warm thank-you message for {name}.",
        expected_output="Thank you message",
        agent=feedback_agent
    )
    crew = Crew(agents=[feedback_agent], tasks=[task])
    return crew.kickoff().raw

def handle_negative_feedback(name, message):
    ticket_id = generate_ticket()
    created_at = str(datetime.datetime.now())

    cursor.execute("INSERT INTO support_tickets VALUES (?, ?, ?, ?, ?)",
                   (ticket_id, name, message, "Unresolved", created_at))
    conn.commit()

    task = Task(
        description=f"Generate apology message including ticket #{ticket_id}.",
        expected_output="Apology message",
        agent=feedback_agent
    )
    crew = Crew(agents=[feedback_agent], tasks=[task])
    return crew.kickoff().raw, ticket_id

def handle_query(message, customer_name):
    match = re.search(r'\d{6}', message)

    if match:
        ticket_id = match.group()
        cursor.execute("SELECT status FROM support_tickets WHERE ticket_id=?", (ticket_id,))
        result = cursor.fetchone()

        if result:
            return f"Your ticket #{ticket_id} is currently marked as: {result[0]}."
        else:
            return f"No ticket found with ID #{ticket_id}."
    else:
        task = Task(
            description=f"Provide professional banking answer for: {message}",
            expected_output="Professional answer",
            agent=query_agent
        )
        crew = Crew(agents=[query_agent], tasks=[task])
        return crew.kickoff().raw

def log_interaction(name, message, classification, agent_name, success=1):
    timestamp = str(datetime.datetime.now())
    cursor.execute("INSERT INTO interaction_logs VALUES (?, ?, ?, ?, ?, ?)",
                   (timestamp, name, message, classification, agent_name, success))
    conn.commit()
